In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")

# Convert timestamp to timezone-aware datetime in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Unique cmdb_id values in each file (compact)
metric_cmdbs = pd.Series(sorted(metric_df['cmdb_id'].dropna().unique()), name='metric_cmdb_ids')
trace_cmdbs = pd.Series(sorted(trace_df['cmdb_id'].dropna().unique()), name='trace_cmdb_ids')

# 2) metric.csv: unique kpi_name values and total row count
metric_kpi_names = pd.Series(sorted(metric_df['kpi_name'].dropna().unique()), name='metric_kpi_names')
metric_total_rows = metric_df.shape[0]

# Aggregation for metric: count, mean, p50, p95, min, max per (cmdb_id, kpi_name)
metric_agg = (
    metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        min='min',
        max='max'
    )
    .reset_index()
)

# Keep results compact: sort by count desc and show top 50, round numeric columns
metric_agg_top = metric_agg.sort_values('count', ascending=False).head(50)
metric_agg_top[['count','mean','p50','p95','min','max']] = metric_agg_top[['count','mean','p50','p95','min','max']].round(6)

# 3) trace.csv: unique trace_name values and total row count
trace_names = pd.Series(sorted(trace_df['trace_name'].dropna().unique()), name='trace_names')
trace_total_rows = trace_df.shape[0]

# Aggregation for trace: count, mean, p50, p95, min, max per (cmdb_id, trace_name)
trace_agg = (
    trace_df.groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        count='count',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        min='min',
        max='max'
    )
    .reset_index()
)

# Keep results compact: sort by count desc and show top 50, round numeric columns
trace_agg_top = trace_agg.sort_values('count', ascending=False).head(50)
trace_agg_top[['count','mean','p50','p95','min','max']] = trace_agg_top[['count','mean','p50','p95','min','max']].round(6)

# Final compact outputs (variables displayed)
metric_cmdbs, trace_cmdbs, metric_kpi_names, metric_total_rows, metric_agg_top, trace_names, trace_total_rows, trace_agg_top

```
Out[1]:
```
```python
# Compose a concise plain-English summary based on the previously computed variables.
# Reuse existing variables from the session: metric_cmdbs, trace_cmdbs, metric_kpi_names,
# metric_total_rows, metric_agg, metric_agg_top, trace_names, trace_total_rows, trace_agg, trace_agg_top

metric_cmdb_count = int(metric_cmdbs.shape[0])
trace_cmdb_count = int(trace_cmdbs.shape[0])
metric_kpi_count = int(metric_kpi_names.shape[0])
trace_name_count = int(trace_names.shape[0])
metric_group_count = int(metric_agg.shape[0])
trace_group_count = int(trace_agg.shape[0])

# Top-group count ranges for compact summary
metric_top_count_min, metric_top_count_max = int(metric_agg_top['count'].min()), int(metric_agg_top['count'].max())
trace_top_count_min, trace_top_count_max = int(trace_agg_top['count'].min()), int(trace_agg_top['count'].max())

# Example top rows (compact fields only)
m_top0 = metric_agg_top.iloc[0][['cmdb_id','kpi_name','count','mean','p95']].to_dict()
t_top0 = trace_agg_top.iloc[0][['cmdb_id','trace_name','count','mean','p95']].to_dict()

summary = (
    "Summary of executed aggregations:\n\n"
    f"1) Unique cmdb_id values found:\n"
    f"   - metric.csv: {metric_cmdb_count} unique cmdb_id values (examples: {metric_cmdbs.head(8).tolist()})\n"
    f"   - trace.csv:  {trace_cmdb_count} unique cmdb_id values (examples: {trace_cmdbs.head(8).tolist()})\n\n"
    f"2) metric.csv overview:\n"
    f"   - Total rows: {metric_total_rows}\n"
    f"   - Unique kpi_name values: {metric_kpi_count} (examples: {metric_kpi_names.head(8).tolist()})\n"
    f"   - Aggregated groups (cmdb_id, kpi_name): {metric_group_count}\n"
    f"   - Top 50 groups (ordered by count) have counts in range [{metric_top_count_min}, {metric_top_count_max}].\n"
    f"   - Example top group: cmdb_id={m_top0['cmdb_id']}, kpi_name={m_top0['kpi_name']}, "
    f"count={int(m_top0['count'])}, mean={m_top0['mean']:.6f}, p95={m_top0['p95']:.6f}\n\n"
    f"3) trace.csv overview:\n"
    f"   - Total rows: {trace_total_rows}\n"
    f"   - Unique trace_name values: {trace_name_count} (examples: {trace_names.head(8).tolist()})\n"
    f"   - Aggregated groups (cmdb_id, trace_name): {trace_group_count}\n"
    f"   - Top 50 groups (ordered by count) have counts in range [{trace_top_count_min}, {trace_top_count_max}].\n"
    f"   - Example top group: cmdb_id={t_top0['cmdb_id']}, trace_name={t_top0['trace_name']}, "
    f"count={int(t_top0['count'])}, mean={t_top0['mean']:.6f}, p95={t_top0['p95']:.6f}\n\n"
    "Notes:\n"
    "- All P95 values were computed on the full series (no time filtering) as requested.\n"
    "- The outputs shown above are compact summaries; full aggregated tables (top 50 rows each) were computed and are available in metric_agg_top and trace_agg_top variables."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
51    redis_008
52    redis_009
53    redis_010
54    redis_011
55    redis_012
Name: metric_cmdb_ids, Length: 56, dtype: object, 0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_006
6    docker_007
7    docker_008
8        os_021
9        os_022
Name: trace_cmdb_ids, dtype: object, 0                   ACS
1                  AIOS
2                   AWS
3            Agent_ping
4           Asm_Free_Tb
             ...       
127        used_cpu_sys
128       used_cpu_user
129         used_memory
130    used_memory_peak
131     used_memory_rss
Name: metric_kpi_names, Length: 132, dtype: object, 90850,       cmdb_id                  kpi_name  count        mean         p50         p95        min         max
1406   os_015  Incoming_network_traffic     61    0.002233    0.002277    0.002352   0.000000    0.003475
1795  osb_001           app.succee_rate     61    0.989058    1.000000    1.000000   0.712950    1.000000
1794  osb_001            app.succee_num     61  472.745902  536.000000  618.000000  14.000000  625.000000
1649   os_020              Disk_io_util     61    0.019308    0.000000    0.005569   0.000000    0.500000
1793  osb_001                   app.num     61  473.073770  536.000000  618.000000  15.000000  625.000000
...       ...                       ...    ...         ...         ...         ...        ...         ...
1610   os_019  Incoming_network_traffic     60    0.027717    0.030923    0.035728   0.002217    0.040828
1624   os_019      Processor_load_5_min     60    0.043444    0.042708    0.053812   0.028333    0.060000
1623   os_019      Processor_load_1_min     60    0.043042    0.039792    0.069250   0.012500    0.109167
1622   os_019     Processor_load_15_min     60    0.041958    0.042083    0.046688   0.036250    0.047083
884    os_005              Disk_io_util     60    0.145695    0.129165    0.333333   0.000000    0.333333

[50 rows x 8 columns], 0     trace.from_docker_001.remoteprocess.duration_mean
1      trace.from_docker_001.remoteprocess.duration_p95
2        trace.from_docker_001.remoteprocess.error_rate
3         trace.from_docker_001.remoteprocess.row_count
4     trace.from_docker_002.remoteprocess.duration_mean
                            ...                        
71          trace.to_docker_007.remoteprocess.row_count
72      trace.to_docker_008.remoteprocess.duration_mean
73       trace.to_docker_008.remoteprocess.duration_p95
74         trace.to_docker_008.remoteprocess.error_rate
75          trace.to_docker_008.remoteprocess.row_count
Name: trace_names, Length: 76, dtype: object, 6240,        cmdb_id                                       trace_name  count         mean          p50          p95         min          max
0   docker_001    trace.from_os_022.remoteprocess.duration_mean     30     3.113841     0.381368    18.937350    0.245270    20.340742
1   docker_001     trace.from_os_022.remoteprocess.duration_p95     30     6.098478     1.160000    28.414450    0.491400    34.222000
2   docker_001       trace.from_os_022.remoteprocess.error_rate     30     0.000000     0.000000     0.000000    0.000000     0.000000
3   docker_001        trace.from_os_022.remoteprocess.row_count     30   126.233333   141.500000   159.750000   31.000000   165.000000
4   docker_001                     trace.self.csf.duration_mean     30     0.108261     0.042492     0.472790    0.032496     0.509000
..         ...                                              ...    ...          ...          ...          ...         ...          ...
45  docker_002                    trace.self.local.duration_p95     30     0.071528     0.081500     0.092292    0.007000     0.094000
46  docker_002                      trace.self.local.error_rate     30     0.000000     0.000000     0.000000    0.000000     0.000000
47  docker_002                       trace.self.local.row_count     30  1804.600000  2023.000000  2239.100000  392.000000  2366.000000
48  docker_002  trace.to_docker_005.remoteprocess.duration_mean     30     0.024437     0.024440     0.025815    0.022333     0.026059
49  docker_002   trace.to_docker_005.remoteprocess.duration_p95     30     0.035645     0.035000     0.038798    0.032650     0.039000

[50 rows x 8 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse existing dataframes if present; otherwise read from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

tz = pytz.timezone('Asia/Shanghai')

# 1) Metric thresholds (global, full series) - compute P95 and P05 per (cmdb_id, kpi_name)
metric_thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
    .reset_index()
)

# Incident window (UTC+8): May 27, 2020 05:00 to 05:30 (inclusive start, exclusive end)
start_ts = tz.localize(pd.to_datetime("2020-05-27 05:00:00"))
end_ts = tz.localize(pd.to_datetime("2020-05-27 05:30:00"))

metric_window = metric_df[(metric_df['ts'] >= start_ts) & (metric_df['ts'] < end_ts)].copy()

# Join thresholds into windowed rows
metric_window = metric_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies: value >= p95 or value <= p05
metric_window['is_anomaly'] = (
    (metric_window['value'] >= metric_window['p95']) |
    (metric_window['value'] <= metric_window['p05'])
)

# Aggregate anomalies by group: count and earliest anomaly timestamp
metric_anoms = (
    metric_window[metric_window['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_ts=('ts', 'min'),
    )
    .reset_index()
)

# Also compute window min/max sample values per group for context (from all window rows)
metric_window_stats = (
    metric_window
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(window_min='min', window_max='max')
    .reset_index()
)

# Join thresholds and window stats into anomaly summary
metric_anom_summary = (
    metric_anoms
    .merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(metric_window_stats, on=['cmdb_id', 'kpi_name'], how='left')
)

# Format earliest timestamp as string for compact display and round numerics
metric_anom_summary['earliest_anomaly_ts'] = metric_anom_summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
metric_anom_summary[['p95','p05','window_min','window_max']] = metric_anom_summary[['p95','p05','window_min','window_max']].round(6)

# Order by anomaly_count desc and take top 20
metric_anom_top = metric_anom_summary.sort_values('anomaly_count', ascending=False).head(20)

# Total number of metric anomalies found in the window
metric_total_anomalies = int(metric_window['is_anomaly'].sum())

# Store thresholds table in kernel as required
metric_thresholds = metric_thresholds.copy()

# 2) Trace thresholds and anomalies (same procedure)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
    .reset_index()
)

trace_window = trace_df[(trace_df['ts'] >= start_ts) & (trace_df['ts'] < end_ts)].copy()
trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
trace_window['is_anomaly'] = (
    (trace_window['value'] >= trace_window['p95']) |
    (trace_window['value'] <= trace_window['p05'])
)

trace_anoms = (
    trace_window[trace_window['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        anomaly_count=('is_anomaly', 'sum'),
        earliest_anomaly_ts=('ts', 'min'),
    )
    .reset_index()
)

trace_window_stats = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(window_min='min', window_max='max')
    .reset_index()
)

trace_anom_summary = (
    trace_anoms
    .merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
    .merge(trace_window_stats, on=['cmdb_id', 'trace_name'], how='left')
)

trace_anom_summary['earliest_anomaly_ts'] = trace_anom_summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_anom_summary[['p95','p05','window_min','window_max']] = trace_anom_summary[['p95','p05','window_min','window_max']].round(6)

trace_anom_top = trace_anom_summary.sort_values('anomaly_count', ascending=False).head(20)
trace_total_anomalies = int(trace_window['is_anomaly'].sum())

# Compact outputs: top 20 metric groups, total metric anomalies, top 20 trace groups, total trace anomalies
metric_anom_top, metric_total_anomalies, trace_anom_top, trace_total_anomalies

```
Out[2]:
```
```python
# Compose a concise plain-English summary using existing variables from the kernel:
# metric_anom_top, metric_total_anomalies, trace_anom_top, trace_total_anomalies

try:
    m_top = metric_anom_top.head(5)[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','p95','p05','window_min','window_max']].to_dict('records')
    t_top = trace_anom_top.head(5)[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','p95','p05','window_min','window_max']].to_dict('records')
    m_total = int(metric_total_anomalies)
    t_total = int(trace_total_anomalies)
except Exception as e:
    summary = "Required summary variables are not available in the kernel."
else:
    def fmt(v):
        if pd.isna(v):
            return "NA"
        # format integers without decimal when large whole numbers, else 6-significant
        if isinstance(v, (int,)) or (isinstance(v, float) and abs(v - round(v)) < 1e-9 and abs(v) > 1e6):
            return str(int(round(v)))
        try:
            return f"{v:.6g}"
        except:
            return str(v)

    lines = []
    lines.append("Incident window: 2020-05-27 05:00 to 05:30 (UTC+8).")
    lines.append("")
    lines.append(f"1) Metrics: total anomaly points in window = {m_total}")
    lines.append("   Top metric groups (up to 5 shown):")
    for r in m_top:
        lines.append(
            f"    - {r['cmdb_id']} | {r['kpi_name']} — anomalies={int(r['anomaly_count'])}, "
            f"earliest={r['earliest_anomaly_ts']}, p95={fmt(r['p95'])}, p05={fmt(r['p05'])}, "
            f"window_min={fmt(r['window_min'])}, window_max={fmt(r['window_max'])}"
        )
    lines.append("")
    lines.append(f"2) Traces: total anomaly points in window = {t_total}")
    lines.append("   Top trace groups (up to 5 shown):")
    for r in t_top:
        lines.append(
            f"    - {r['cmdb_id']} | {r.get('trace_name')} — anomalies={int(r['anomaly_count'])}, "
            f"earliest={r['earliest_anomaly_ts']}, p95={fmt(r['p95'])}, p05={fmt(r['p05'])}, "
            f"window_min={fmt(r['window_min'])}, window_max={fmt(r['window_max'])}"
        )
    lines.append("")
    lines.append("Notes:")
    lines.append("- P95/P05 thresholds were computed on the full available series before filtering to the window.")
    lines.append("- Many top metric anomalies are from redis nodes (e.g., redis_012, redis_010, redis_009) with full-window constant values during the window.")
    lines.append("- Many top trace anomalies are error_rate or duration p95-related metrics from docker_xxx services, with earliest anomalies at the window start.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                 kpi_name  anomaly_count      earliest_anomaly_ts           p95           p05    window_min    window_max
1974  redis_012          used_memory_rss             30  2020-05-27 05:00:00 CST  2.417275e+09  2.417275e+09  2.417275e+09  2.417275e+09
1973  redis_012         used_memory_peak             30  2020-05-27 05:00:00 CST  3.112927e+09  3.112927e+09  3.112927e+09  3.112927e+09
1        db_001                     AIOS             30  2020-05-27 05:00:00 CST  1.000000e-02  0.000000e+00  0.000000e+00  1.000000e-02
1967  redis_012     rejected_connections             30  2020-05-27 05:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1966  redis_012               redis_ping             30  2020-05-27 05:00:00 CST  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
1965  redis_012               redis_load             30  2020-05-27 05:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1964  redis_012  mem_fragmentation_ratio             30  2020-05-27 05:00:00 CST  9.300000e-01  9.300000e-01  9.300000e-01  9.300000e-01
1963  redis_012                maxmemory             30  2020-05-27 05:00:00 CST  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09
1962  redis_012          keyspace_misses             30  2020-05-27 05:00:00 CST  1.320700e+04  1.320700e+04  1.320700e+04  1.320700e+04
1961  redis_012            keyspace_hits             30  2020-05-27 05:00:00 CST  1.478380e+05  1.478380e+05  1.478380e+05  1.478380e+05
1922  redis_010               redis_load             30  2020-05-27 05:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1923  redis_010               redis_ping             30  2020-05-27 05:00:00 CST  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
1924  redis_010     rejected_connections             30  2020-05-27 05:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1914  redis_010             expired_keys             30  2020-05-27 05:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1918  redis_010            keyspace_hits             30  2020-05-27 05:00:00 CST  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
1919  redis_010          keyspace_misses             30  2020-05-27 05:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1920  redis_010                maxmemory             30  2020-05-27 05:00:00 CST  2.000000e+09  2.000000e+09  2.000000e+09  2.000000e+09
1908  redis_009         used_memory_peak             30  2020-05-27 05:00:00 CST  3.625788e+07  3.625788e+07  3.625788e+07  3.625788e+07
1910  redis_010          Redis_key_count             30  2020-05-27 05:00:00 CST  1.981200e+04  1.981200e+04  1.981200e+04  1.981200e+04
1911  redis_010          blocked_clients             30  2020-05-27 05:00:00 CST  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00, 24087,        cmdb_id                                    trace_name  anomaly_count      earliest_anomaly_ts    p95    p05  window_min  window_max
2   docker_001    trace.from_os_022.remoteprocess.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
6   docker_001                     trace.self.csf.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
14  docker_001                    trace.self.jdbc.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
10  docker_001               trace.self.flyremote.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
74  docker_003                   trace.self.local.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
70  docker_003                    trace.self.jdbc.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
69  docker_003                  trace.self.jdbc.duration_p95             30  2020-05-27 05:00:00 CST  0.004  0.003       0.003       0.012
46  docker_002                   trace.self.local.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
34  docker_002                     trace.self.csf.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
38  docker_002               trace.self.flyremote.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
42  docker_002                    trace.self.jdbc.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
41  docker_002                  trace.self.jdbc.duration_p95             30  2020-05-27 05:00:00 CST  0.003  0.003       0.003       0.003
26  docker_001  trace.to_docker_006.remoteprocess.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
30  docker_002    trace.from_os_022.remoteprocess.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
22  docker_001  trace.to_docker_005.remoteprocess.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
18  docker_001                   trace.self.local.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
66  docker_003               trace.self.flyremote.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
78  docker_003  trace.to_docker_007.remoteprocess.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
54  docker_002  trace.to_docker_006.remoteprocess.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000
50  docker_002  trace.to_docker_005.remoteprocess.error_rate             30  2020-05-27 05:00:00 CST  0.000  0.000       0.000       0.000, 2486)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing in-memory variables: metric_window, metric_anom_top, metric_thresholds, metric_window_stats, metric_anom_summary
# If any are missing, load / recompute minimal needed parts.
tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_window and metric_anom_top exist
try:
    metric_window
    metric_anom_top
    metric_thresholds
    metric_window_stats
except NameError:
    # Minimal recompute if needed (should not normally happen in the session)
    metric_df = pd.read_csv("metric.csv")
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_thresholds = (
        metric_df.groupby(['cmdb_id','kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
        .reset_index()
    )
    start_ts = tz.localize(pd.to_datetime("2020-05-27 05:00:00"))
    end_ts = tz.localize(pd.to_datetime("2020-05-27 05:30:00"))
    metric_window = metric_df[(metric_df['ts'] >= start_ts) & (metric_df['ts'] < end_ts)].copy()
    metric_window = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    metric_window['is_anomaly'] = (metric_window['value'] >= metric_window['p95']) | (metric_window['value'] <= metric_window['p05'])
    metric_window_stats = metric_window.groupby(['cmdb_id','kpi_name'])['value'].agg(window_min='min', window_max='max').reset_index()
    metric_anoms = (
        metric_window[metric_window['is_anomaly']]
        .groupby(['cmdb_id','kpi_name'])
        .agg(anomaly_count=('is_anomaly','sum'), earliest_anomaly_ts=('ts','min'))
        .reset_index()
    )
    metric_anom_summary = metric_anoms.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left').merge(metric_window_stats, on=['cmdb_id','kpi_name'], how='left')
    metric_anom_summary['earliest_anomaly_ts'] = metric_anom_summary['earliest_anomaly_ts'].dt.tz_convert(tz)
    metric_anom_top = metric_anom_summary.sort_values('anomaly_count', ascending=False).head(20)

# Work on top 20 groups ordered by anomaly_count (use metric_anom_top as source)
top_groups = metric_anom_top[['cmdb_id','kpi_name']].drop_duplicates().head(20)

# Filter metric_window to anomalies within the window for these groups
mw = metric_window.merge(top_groups.assign(_merge_key=1), on=['cmdb_id','kpi_name'], how='right')
# Now mw contains only rows for top groups (including non-anomaly rows) -- keep original is_anomaly marker
# We'll analyze only anomaly rows for runs, but need window_min/max from all window rows for the group
results = []

for (cmdb, kpi), grp in mw.groupby(['cmdb_id','kpi_name']):
    # All window rows for this group
    grp_all = grp.sort_values('ts')
    # Anomalous rows only
    grp_anom = grp_all[grp_all['is_anomaly']].sort_values('ts').copy()
    total_anomaly_count = int(grp_anom.shape[0])
    if total_anomaly_count == 0:
        # Shouldn't happen for metric_anom_top, but guard anyway
        earliest_ts = pd.NaT
        num_segments = 0
        longest_run = 0
    else:
        earliest_ts = grp_anom['ts'].min()
        # compute consecutive segments: difference in minutes between consecutive anomaly timestamps
        diffs_min = grp_anom['ts'].diff().dt.total_seconds().div(60)
        # start of a new segment where diff != 1 or first element (NaN)
        new_segment = (diffs_min != 1) | (diffs_min.isna())
        # label segments by cumulative sum of new_segment
        seg_label = new_segment.cumsum()
        seg_sizes = grp_anom.groupby(seg_label).size().to_numpy()
        num_segments = int(seg_sizes.size)
        longest_run = int(seg_sizes.max())
    # window min/max from grp_all
    window_min = float(grp_all['value'].min()) if not grp_all['value'].isna().all() else np.nan
    window_max = float(grp_all['value'].max()) if not grp_all['value'].isna().all() else np.nan
    # thresholds p95/p05 (present in joined metric_window rows)
    p95 = float(grp_all['p95'].iloc[0]) if 'p95' in grp_all.columns else np.nan
    p05 = float(grp_all['p05'].iloc[0]) if 'p05' in grp_all.columns else np.nan
    # severity calculation
    high_breach = 0.0
    low_breach = 0.0
    # high-side: if window_max >= p95
    if not np.isnan(window_max) and not np.isnan(p95) and window_max >= p95:
        high_breach = (window_max - p95) / (abs(p95) + 1e-12)
    # low-side: if window_min <= p05
    if not np.isnan(window_min) and not np.isnan(p05) and window_min <= p05:
        low_breach = (p05 - window_min) / (abs(p05) + 1e-12)
    severity_score = float(max(high_breach, low_breach))
    results.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'total_anomaly_count': total_anomaly_count,
        'earliest_anomaly_ts': earliest_ts,
        'number_of_consecutive_anomaly_segments': num_segments,
        'longest_consecutive_run_length': longest_run,
        'window_min': window_min,
        'window_max': window_max,
        'p95': p95,
        'p05': p05,
        'severity_score': severity_score
    })

# Build DataFrame
metric_faults = pd.DataFrame(results)

# Format earliest_anomaly_ts as string in UTC+8 for compact display
metric_faults['earliest_anomaly_ts'] = metric_faults['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Round numeric columns for compactness
for c in ['window_min','window_max','p95','p05','severity_score']:
    metric_faults[c] = metric_faults[c].map(lambda v: round(v,6) if pd.notna(v) else v)

# Order by severity_score desc then total_anomaly_count desc, limit to top 20
metric_faults_top20 = metric_faults.sort_values(['severity_score','total_anomaly_count'], ascending=[False, False]).head(20)

# Keep output compact: select requested columns
metric_faults_top20 = metric_faults_top20[[
    'cmdb_id','kpi_name','total_anomaly_count','earliest_anomaly_ts',
    'number_of_consecutive_anomaly_segments','longest_consecutive_run_length',
    'window_min','window_max','p95','p05','severity_score'
]]

metric_faults_top20

```
Out[3]:
```
```python
# Compose a plain-English summary string using existing kernel variables and display it.
unique_counts = metric_faults_top20['total_anomaly_count'].unique()
unique_segments = metric_faults_top20['number_of_consecutive_anomaly_segments'].unique()
unique_longest = metric_faults_top20['longest_consecutive_run_length'].unique()
severity_max = metric_faults_top20['severity_score'].max()
example_rows = metric_faults_top20.head(5).to_dict('records')

summary = (
    "Summary of top-20 metric groups (incident window 2020-05-27 05:00–05:30 UTC+8):\n\n"
    f"- Total metric anomaly points in the window (all metrics): {metric_total_anomalies}\n"
    f"- Top-20 groups examined: {len(metric_faults_top20)} groups.\n\n"
    "Observed patterns across the top-20 groups:\n"
    f"- total_anomaly_count: mostly {unique_counts.tolist()} (each top group shows 30 anomalies).\n"
    f"- earliest_anomaly_ts: for top groups the earliest anomaly is {metric_faults_top20['earliest_anomaly_ts'].unique()[0]} (window start).\n"
    f"- number_of_consecutive_anomaly_segments: {unique_segments.tolist()} (all groups form a single continuous segment in the window).\n"
    f"- longest_consecutive_run_length (minutes): {unique_longest.tolist()} (runs span the full 30 minutes for these groups).\n"
    f"- window_min/window_max vs p95/p05: for many top groups window_min == window_max == p95 == p05 (i.e., constant values during the window).\n"
    f"- severity_score: max = {severity_max}. For the top-20 groups the severity_score is 0.0 (no magnitude breach beyond the threshold; values equal thresholds).\n\n"
    "Interpretation:\n"
    "- The top-20 metric groups are constant-valued across the incident window and flagged as anomalies because their values equal the computed global thresholds (P95/P05), producing continuous 30-minute anomaly runs but zero severity by the breach-magnitude metric.\n"
    "- This pattern suggests either true sustained state changes (constant extreme values) or a sampling/measurement artifact that produced identical values for the window. Investigate metric collection and the raw time series for these redis nodes (e.g., redis_012, redis_010, redis_009) to confirm.\n\n"
    "Example (first 5 of top-20):\n"
)

for r in example_rows:
    summary += (
        f" - {r['cmdb_id']} | {r['kpi_name']} — anomalies={r['total_anomaly_count']}, "
        f"earliest={r['earliest_anomaly_ts']}, segments={r['number_of_consecutive_anomaly_segments']}, "
        f"longest_run={r['longest_consecutive_run_length']}, window_min={r['window_min']}, window_max={r['window_max']}, "
        f"p95={r['p95']}, p05={r['p05']}, severity={r['severity_score']}\n"
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                 kpi_name  total_anomaly_count      earliest_anomaly_ts  number_of_consecutive_anomaly_segments  longest_consecutive_run_length    window_min    window_max           p95           p05  severity_score
0      db_001                     AIOS                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  1.000000e-02  1.000000e-02  0.000000e+00             0.0
1   redis_009         used_memory_peak                   30  2020-05-27 05:00:00 CST                                       1                              30  3.625788e+07  3.625788e+07  3.625788e+07  3.625788e+07             0.0
2   redis_010          Redis_key_count                   30  2020-05-27 05:00:00 CST                                       1                              30  1.981200e+04  1.981200e+04  1.981200e+04  1.981200e+04             0.0
3   redis_010          blocked_clients                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             0.0
4   redis_010             expired_keys                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             0.0
5   redis_010            keyspace_hits                   30  2020-05-27 05:00:00 CST                                       1                              30  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00             0.0
6   redis_010          keyspace_misses                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             0.0
7   redis_010                maxmemory                   30  2020-05-27 05:00:00 CST                                       1                              30  2.000000e+09  2.000000e+09  2.000000e+09  2.000000e+09             0.0
8   redis_010               redis_load                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             0.0
9   redis_010               redis_ping                   30  2020-05-27 05:00:00 CST                                       1                              30  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00             0.0
10  redis_010     rejected_connections                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             0.0
11  redis_012            keyspace_hits                   30  2020-05-27 05:00:00 CST                                       1                              30  1.478380e+05  1.478380e+05  1.478380e+05  1.478380e+05             0.0
12  redis_012          keyspace_misses                   30  2020-05-27 05:00:00 CST                                       1                              30  1.320700e+04  1.320700e+04  1.320700e+04  1.320700e+04             0.0
13  redis_012                maxmemory                   30  2020-05-27 05:00:00 CST                                       1                              30  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09             0.0
14  redis_012  mem_fragmentation_ratio                   30  2020-05-27 05:00:00 CST                                       1                              30  9.300000e-01  9.300000e-01  9.300000e-01  9.300000e-01             0.0
15  redis_012               redis_load                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             0.0
16  redis_012               redis_ping                   30  2020-05-27 05:00:00 CST                                       1                              30  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00             0.0
17  redis_012     rejected_connections                   30  2020-05-27 05:00:00 CST                                       1                              30  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00             0.0
18  redis_012         used_memory_peak                   30  2020-05-27 05:00:00 CST                                       1                              30  3.112927e+09  3.112927e+09  3.112927e+09  3.112927e+09             0.0
19  redis_012          used_memory_rss                   30  2020-05-27 05:00:00 CST                                       1                              30  2.417275e+09  2.417275e+09  2.417275e+09  2.417275e+09             0.0```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables if present; otherwise recompute necessary data
try:
    trace_window
    trace_anom_top
    trace_thresholds
except NameError:
    # load and compute minimal needed pieces
    trace_df = pd.read_csv("trace.csv")
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    trace_thresholds = (
        trace_df.groupby(['cmdb_id','trace_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
        .reset_index()
    )
    start_ts = tz.localize(pd.to_datetime("2020-05-27 05:00:00"))
    end_ts = tz.localize(pd.to_datetime("2020-05-27 05:30:00"))
    trace_window = trace_df[(trace_df['ts'] >= start_ts) & (trace_df['ts'] < end_ts)].copy()
    trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    trace_window['is_anomaly'] = (trace_window['value'] >= trace_window['p95']) | (trace_window['value'] <= trace_window['p05'])
    trace_anoms = (
        trace_window[trace_window['is_anomaly']]
        .groupby(['cmdb_id','trace_name'])
        .agg(anomaly_count=('is_anomaly','sum'), earliest_anomaly_ts=('ts','min'))
        .reset_index()
    )
    trace_window_stats = trace_window.groupby(['cmdb_id','trace_name'])['value'].agg(window_min='min', window_max='max').reset_index()
    trace_anom_top = (
        trace_anoms
        .merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
        .merge(trace_window_stats, on=['cmdb_id','trace_name'], how='left')
        .sort_values('anomaly_count', ascending=False)
        .head(20)
    )

# Use top 20 trace groups by anomaly_count from trace_anom_top
top_trace_groups = trace_anom_top[['cmdb_id','trace_name']].drop_duplicates().head(20)

# Filter trace_window to only these groups
tw = trace_window.merge(top_trace_groups.assign(_k=1), on=['cmdb_id','trace_name'], how='right').copy()

results = []
for (cmdb, trace_name), grp in tw.groupby(['cmdb_id','trace_name']):
    grp_all = grp.sort_values('ts')
    # anomaly rows only
    grp_anom = grp_all[grp_all['is_anomaly']].sort_values('ts').copy()
    total_anomaly_count = int(grp_anom.shape[0])
    if total_anomaly_count == 0:
        earliest = pd.NaT
        num_segments = 0
        longest_run = 0
    else:
        earliest = grp_anom['ts'].min()
        diffs_min = grp_anom['ts'].diff().dt.total_seconds().div(60)
        # start of new segment where diff != 1 or first element
        new_seg = (diffs_min != 1) | (diffs_min.isna())
        seg_label = new_seg.cumsum()
        seg_sizes = grp_anom.groupby(seg_label).size().to_numpy()
        num_segments = int(seg_sizes.size)
        longest_run = int(seg_sizes.max())
    window_min = float(grp_all['value'].min()) if not grp_all['value'].isna().all() else np.nan
    window_max = float(grp_all['value'].max()) if not grp_all['value'].isna().all() else np.nan
    p95 = float(grp_all['p95'].iloc[0]) if 'p95' in grp_all.columns else np.nan
    p05 = float(grp_all['p05'].iloc[0]) if 'p05' in grp_all.columns else np.nan
    high_breach = 0.0
    low_breach = 0.0
    if not np.isnan(window_max) and not np.isnan(p95) and window_max >= p95:
        high_breach = (window_max - p95) / (abs(p95) + 1e-12)
    if not np.isnan(window_min) and not np.isnan(p05) and window_min <= p05:
        low_breach = (p05 - window_min) / (abs(p05) + 1e-12)
    severity_score = float(max(high_breach, low_breach))
    results.append({
        'cmdb_id': cmdb,
        'trace_name': trace_name,
        'total_anomaly_count': total_anomaly_count,
        'earliest_anomaly_ts': earliest,
        'number_of_consecutive_anomaly_segments': num_segments,
        'longest_consecutive_run_length': longest_run,
        'window_min': window_min,
        'window_max': window_max,
        'p95': p95,
        'p05': p05,
        'severity_score': severity_score
    })

trace_faults = pd.DataFrame(results)

# Format and round for compact display
trace_faults['earliest_anomaly_ts'] = trace_faults['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
for c in ['window_min','window_max','p95','p05','severity_score']:
    trace_faults[c] = trace_faults[c].map(lambda v: round(v,6) if pd.notna(v) else v)

# Order by severity_score desc then total_anomaly_count desc, limit to top 20
trace_faults_top20 = trace_faults.sort_values(['severity_score','total_anomaly_count'], ascending=[False, False]).head(20)

# Display the compact top-20 table
trace_faults_top20

```
Out[4]:
```
```python
# Compose a concise plain-English summary using existing kernel variables: trace_faults_top20 and trace_total_anomalies.
try:
    tbl = trace_faults_top20
    total = int(trace_total_anomalies)
except NameError:
    summary = "Required variables (trace_faults_top20 or trace_total_anomalies) are not available in the kernel."
else:
    top_example = tbl.head(5).to_dict('records')
    # Determine max severity and how many non-zero severity rows
    max_sev = float(tbl['severity_score'].max())
    nonzero_count = int((tbl['severity_score'] > 0).sum())
    # Most groups properties (common patterns)
    common_anomaly_count = int(tbl['total_anomaly_count'].mode()[0]) if not tbl.empty else None
    earliest_examples = sorted(tbl['earliest_anomaly_ts'].unique())
    lines = []
    lines.append("Trace anomaly fault-identification summary (window 2020-05-27 05:00–05:30 UTC+8):")
    lines.append("")
    lines.append(f"- Total trace anomaly points in the window: {total}")
    lines.append(f"- Top {len(tbl)} trace groups analyzed (by anomaly_count).")
    lines.append("")
    lines.append("Key findings:")
    lines.append(f"- Most top groups have total_anomaly_count = {common_anomaly_count} (commonly 30) and earliest_anomaly_ts = {earliest_examples[0] if earliest_examples.size else 'N/A'} (window start).")
    lines.append(f"- Consecutive anomaly behavior: groups typically form a single continuous segment covering the full 30-minute window (one segment, longest_run = 30).")
    lines.append(f"- Severity: max severity_score = {max_sev}; non-zero severity groups = {nonzero_count}.")
    lines.append("")
    lines.append("Notable group:")
    # Find row(s) with max severity
    if max_sev > 0:
        max_rows = tbl[tbl['severity_score'] == max_sev].to_dict('records')
        for r in max_rows[:3]:
            lines.append(
                f" - {r['cmdb_id']} | {r['trace_name']} — anomalies={r['total_anomaly_count']}, "
                f"earliest={r['earliest_anomaly_ts']}, window_min={r['window_min']}, window_max={r['window_max']}, "
                f"p95={r['p95']}, p05={r['p05']}, severity={r['severity_score']}"
            )
    else:
        lines.append(" - No significant magnitude breaches detected among top groups (severity_score = 0 for most).")
    lines.append("")
    lines.append("Interpretation & next steps:")
    lines.append("- Many top trace groups show constant or saturated values across the window and were flagged because they match global thresholds (P95/P05 computed on full series).")
    lines.append("- One group (docker_003 | trace.self.jdbc.duration_p95) shows a clear high-side breach (severity=2.0) and should be prioritized for investigation (duration spike).")
    lines.append("- Recommend inspecting raw traces and related logs for the highlighted docker_003 JDBC p95 spike, and verify metric/trace collection for constant-valued groups to rule out instrumentation issues.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  total_anomaly_count      earliest_anomaly_ts  number_of_consecutive_anomaly_segments  longest_consecutive_run_length  window_min  window_max    p95    p05  severity_score
16  docker_003                  trace.self.jdbc.duration_p95                   30  2020-05-27 05:00:00 CST                                       1                              30       0.003       0.012  0.004  0.003             2.0
0   docker_001    trace.from_os_022.remoteprocess.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
1   docker_001                     trace.self.csf.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
2   docker_001               trace.self.flyremote.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
3   docker_001                    trace.self.jdbc.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
4   docker_001                   trace.self.local.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
5   docker_001  trace.to_docker_005.remoteprocess.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
6   docker_001  trace.to_docker_006.remoteprocess.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
7   docker_002    trace.from_os_022.remoteprocess.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
8   docker_002                     trace.self.csf.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
9   docker_002               trace.self.flyremote.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
10  docker_002                  trace.self.jdbc.duration_p95                   30  2020-05-27 05:00:00 CST                                       1                              30       0.003       0.003  0.003  0.003             0.0
11  docker_002                    trace.self.jdbc.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
12  docker_002                   trace.self.local.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
13  docker_002  trace.to_docker_005.remoteprocess.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
14  docker_002  trace.to_docker_006.remoteprocess.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
15  docker_003               trace.self.flyremote.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
17  docker_003                    trace.self.jdbc.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
18  docker_003                   trace.self.local.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0
19  docker_003  trace.to_docker_007.remoteprocess.error_rate                   30  2020-05-27 05:00:00 CST                                       1                              30       0.000       0.000  0.000  0.000             0.0```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing windowed trace data if present, otherwise load and prepare
try:
    trace_window
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    start_ts = tz.localize(pd.to_datetime("2020-05-27 05:00:00"))
    end_ts = tz.localize(pd.to_datetime("2020-05-27 05:30:00"))
    trace_window = trace_df[(trace_df['ts'] >= start_ts) & (trace_df['ts'] < end_ts)].copy()
else:
    # ensure ts exists and is timezone-aware
    if 'ts' not in trace_window.columns:
        trace_window['ts'] = pd.to_datetime(trace_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Aggregated summary by trace_name for cmdb_id == 'docker_003'
docker3_window = trace_window[trace_window['cmdb_id'] == 'docker_003'].copy()

agg_summary = (
    docker3_window
    .groupby('trace_name')['value']
    .agg(
        count='count',
        earliest_ts=lambda x: docker3_window.loc[x.index, 'ts'].min(),
        window_min='min',
        window_max='max',
        mean='mean'
    )
    .reset_index()
)

# Format earliest_ts as string in UTC+8 and round numeric columns
agg_summary['earliest_ts'] = pd.to_datetime(agg_summary['earliest_ts']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
for c in ['window_min','window_max','mean']:
    agg_summary[c] = agg_summary[c].round(6)

# Order by count desc and limit to top 20
agg_summary = agg_summary.sort_values('count', ascending=False).head(20)
agg_summary = agg_summary[['trace_name','count','earliest_ts','window_min','window_max','mean']]

# 2) Raw rows for trace_name == 'trace.self.jdbc.duration_p95' (timestamp, trace_name, value), ordered asc, top 50
raw_rows = (
    docker3_window[docker3_window['trace_name'] == 'trace.self.jdbc.duration_p95']
    .sort_values('timestamp')[['timestamp','trace_name','value']]
    .head(50)
)

# Compact outputs
agg_summary, raw_rows

```
Out[5]:
```
```python
# Summarize docker_003 trace-window results using existing variables: agg_summary and raw_rows
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Safety: ensure variables exist
try:
    agg_summary
    raw_rows
except NameError:
    summary = "Required data (agg_summary or raw_rows) is not available in the kernel."
else:
    num_groups = int(agg_summary.shape[0])
    common_count = int(agg_summary['count'].mode()[0]) if not agg_summary.empty else 0
    earliest_ts = agg_summary['earliest_ts'].unique().tolist()
    top5 = agg_summary.head(5).to_dict('records')
    # Details for trace.self.jdbc.duration_p95
    target = 'trace.self.jdbc.duration_p95'
    raw_target = raw_rows.copy()
    raw_count = int(raw_target.shape[0])
    val_counts = raw_target['value'].value_counts().to_dict()
    # find any spike timestamps for values > typical
    spikes = raw_target[raw_target['value'] > raw_target['value'].min()]
    spike_info = []
    for _, row in spikes.iterrows():
        ts_human = pd.to_datetime(row['timestamp'], unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        spike_info.append((row['value'], ts_human))
    # earliest / latest
    if raw_count > 0:
        earliest_raw = pd.to_datetime(raw_target['timestamp'].min(), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        latest_raw = pd.to_datetime(raw_target['timestamp'].max(), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_raw = latest_raw = "N/A"

    lines = []
    lines.append("docker_003 — trace data (window 2020-05-27 05:00–05:30 UTC+8):")
    lines.append("")
    lines.append(f"1) Aggregated summary by trace_name (top {num_groups} shown):")
    lines.append(f"   - Number of trace_name groups returned: {num_groups}")
    lines.append(f"   - Per-group row counts are uniform: most groups have count = {common_count} (one-minute samples across the 30-minute window).")
    lines.append(f"   - Earliest sample for groups: {earliest_ts[0] if earliest_ts else 'N/A'} (window start).")
    lines.append("   - Example top 5 trace_name aggregates (trace_name, count, earliest_ts, window_min, window_max, mean):")
    for r in top5:
        lines.append(f"     • {r['trace_name']} — count={r['count']}, earliest={r['earliest_ts']}, min={r['window_min']}, max={r['window_max']}, mean={r['mean']}")
    lines.append("")
    lines.append("2) Raw rows for trace_name == 'trace.self.jdbc.duration_p95':")
    lines.append(f"   - Total raw rows in window: {raw_count} (max 30 expected for 30 minutes).")
    lines.append(f"   - Value distribution (value: occurrences): {val_counts}")
    lines.append(f"   - Earliest timestamp in raw rows: {earliest_raw}")
    lines.append(f"   - Latest timestamp in raw rows:   {latest_raw}")
    if spike_info:
        # show up to first 5 spike examples
        spike_examples = spike_info[:5]
        stext = ", ".join([f"{v}@{t}" for v, t in spike_examples])
        lines.append(f"   - Notable higher-than-min values (value@timestamp): {stext}")
    else:
        lines.append("   - No values above the minimum value found in the raw sample set.")
    lines.append("")
    lines.append("Quick interpretation:")
    lines.append(" - docker_003 has consistent per-minute trace samples across the window (most groups show 30 samples).")
    lines.append(" - The JDBC p95 trace shows a stable baseline at 0.003 with occasional brief increases (e.g., 0.004 and a single 0.012 spike).")
    lines.append(" - Investigate the timestamps of the higher values (listed above) if you need to correlate with logs or other events.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                       trace_name  count              earliest_ts  window_min   window_max         mean
0   trace.from_os_021.remoteprocess.duration_mean     30  2020-05-27 05:00:00 CST    0.332688     0.801685     0.568551
1    trace.from_os_021.remoteprocess.duration_p95     30  2020-05-27 05:00:00 CST    0.913300     2.412400     1.554222
2      trace.from_os_021.remoteprocess.error_rate     30  2020-05-27 05:00:00 CST    0.000000     0.000000     0.000000
3       trace.from_os_021.remoteprocess.row_count     30  2020-05-27 05:00:00 CST   29.000000   169.000000   127.633333
4                    trace.self.csf.duration_mean     30  2020-05-27 05:00:00 CST    0.038129     0.074628     0.055890
5                     trace.self.csf.duration_p95     30  2020-05-27 05:00:00 CST    0.098050     0.193850     0.143272
6                       trace.self.csf.error_rate     30  2020-05-27 05:00:00 CST    0.000000     0.000000     0.000000
7                        trace.self.csf.row_count     30  2020-05-27 05:00:00 CST  116.000000   676.000000   510.533333
8              trace.self.flyremote.duration_mean     30  2020-05-27 05:00:00 CST    0.007889     0.016314     0.011173
9               trace.self.flyremote.duration_p95     30  2020-05-27 05:00:00 CST    0.007000     0.098300     0.054410
10                trace.self.flyremote.error_rate     30  2020-05-27 05:00:00 CST    0.000000     0.000000     0.000000
11                 trace.self.flyremote.row_count     30  2020-05-27 05:00:00 CST   29.000000   169.000000   127.633333
12                  trace.self.jdbc.duration_mean     30  2020-05-27 05:00:00 CST    0.001875     0.004531     0.003307
13                   trace.self.jdbc.duration_p95     30  2020-05-27 05:00:00 CST    0.003000     0.012000     0.003467
14                     trace.self.jdbc.error_rate     30  2020-05-27 05:00:00 CST    0.000000     0.000000     0.000000
15                      trace.self.jdbc.row_count     30  2020-05-27 05:00:00 CST  464.000000  2708.000000  2044.300000
16                 trace.self.local.duration_mean     30  2020-05-27 05:00:00 CST    0.005614     0.018630     0.013115
17                  trace.self.local.duration_p95     30  2020-05-27 05:00:00 CST    0.009000     0.097000     0.084343
18                    trace.self.local.error_rate     30  2020-05-27 05:00:00 CST    0.000000     0.000000     0.000000
19                     trace.self.local.row_count     30  2020-05-27 05:00:00 CST  406.000000  2366.000000  1786.866667,        timestamp                    trace_name  value
69    1590526800  trace.self.jdbc.duration_p95  0.003
277   1590526860  trace.self.jdbc.duration_p95  0.004
485   1590526920  trace.self.jdbc.duration_p95  0.003
693   1590526980  trace.self.jdbc.duration_p95  0.003
901   1590527040  trace.self.jdbc.duration_p95  0.003
...          ...                           ...    ...
5269  1590528300  trace.self.jdbc.duration_p95  0.003
5477  1590528360  trace.self.jdbc.duration_p95  0.003
5685  1590528420  trace.self.jdbc.duration_p95  0.003
5893  1590528480  trace.self.jdbc.duration_p95  0.004
6101  1590528540  trace.self.jdbc.duration_p95  0.003

[30 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_anom_summary and metric_window exist; otherwise recompute minimally
try:
    metric_anom_summary
    metric_window
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_thresholds = (
        metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
        .reset_index()
    )
    start_ts = tz.localize(pd.to_datetime("2020-05-27 05:00:00"))
    end_ts = tz.localize(pd.to_datetime("2020-05-27 05:30:00"))
    metric_window = metric_df[(metric_df['ts'] >= start_ts) & (metric_df['ts'] < end_ts)].copy()
    metric_window = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    metric_window['is_anomaly'] = (metric_window['value'] >= metric_window['p95']) | (metric_window['value'] <= metric_window['p05'])
    metric_anoms = (
        metric_window[metric_window['is_anomaly']]
        .groupby(['cmdb_id','kpi_name'])
        .agg(anomaly_count=('is_anomaly','sum'), earliest_anomaly_ts=('ts','min'))
        .reset_index()
    )
    metric_window_stats = metric_window.groupby(['cmdb_id','kpi_name'])['value'].agg(window_min='min', window_max='max').reset_index()
    metric_anom_summary = (
        metric_anoms
        .merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
        .merge(metric_window_stats, on=['cmdb_id','kpi_name'], how='left')
    )

# Filter to cmdb_id == 'docker_003' OR startswith 'db_'
mask = (metric_anom_summary['cmdb_id'] == 'docker_003') | metric_anom_summary['cmdb_id'].astype(str).str.startswith('db_')
subset = metric_anom_summary[mask].copy()

# Normalize earliest_anomaly_ts to datetime-aware (UTC+8) if not already
if subset.shape[0] > 0:
    # parse if not datetimelike
    if not pd.api.types.is_datetime64_any_dtype(subset['earliest_anomaly_ts']):
        subset['earliest_anomaly_ts'] = pd.to_datetime(subset['earliest_anomaly_ts'], errors='coerce', utc=True).dt.tz_convert(tz)
    else:
        # ensure tz-aware and convert
        subset['earliest_anomaly_ts'] = pd.to_datetime(subset['earliest_anomaly_ts']).dt.tz_localize(None).dt.tz_localize('UTC').dt.tz_convert(tz)

# Compute severity_score
def compute_severity(row):
    wm = row.get('window_max', np.nan)
    wn = row.get('window_min', np.nan)
    p95 = row.get('p95', np.nan)
    p05 = row.get('p05', np.nan)
    high = 0.0
    low = 0.0
    if pd.notna(wm) and pd.notna(p95) and wm >= p95:
        high = (wm - p95) / (abs(p95) + 1e-12)
    if pd.notna(wn) and pd.notna(p05) and wn <= p05:
        low = (p05 - wn) / (abs(p05) + 1e-12)
    return max(high, low)

subset['severity_score'] = subset.apply(compute_severity, axis=1)

# Format earliest timestamp string and round numerics
subset['earliest_anomaly_ts'] = subset['earliest_anomaly_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
for c in ['window_min','window_max','p95','p05','severity_score']:
    subset[c] = subset[c].apply(lambda v: round(float(v),6) if pd.notna(v) else v)

# 1) Per-(cmdb_id,kpi_name) rows with severity_score > 0, ordered and limited to top 20
per_group = subset[subset['severity_score'] > 0].copy()
per_group = per_group[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','window_min','window_max','p95','p05','severity_score']]
per_group = per_group.sort_values(['severity_score','anomaly_count'], ascending=[False, False]).head(20)

# 2) Per-component summary: sum anomaly_count by cmdb_id for selected components
per_component = (
    subset.groupby('cmdb_id', as_index=False)['anomaly_count']
    .sum()
    .rename(columns={'anomaly_count':'total_anomalies'})
    .sort_values('total_anomalies', ascending=False)
)

# Return compact outputs (per_group top20, all per_component rows)
per_group, per_component

```
Out[6]:
```
```python
# Compose a concise plain-English summary using the computed tables: per_group and per_component
try:
    per_group, per_component
except NameError:
    summary = "Required results (per_group or per_component) are not available in the kernel."
else:
    top_n = min(5, per_group.shape[0])
    top_examples = per_group.head(top_n).to_dict('records')
    rows_shown = per_group.shape[0]
    comp_rows = per_component.shape[0]
    # Build summary lines
    lines = []
    lines.append("Metric anomaly fault-identification (filtered to cmdb_id == 'docker_003' OR cmdb_id starts with 'db_'):")
    lines.append("")
    lines.append(f"1) Per-(cmdb_id, kpi_name) with severity_score > 0: {rows_shown} rows (top {min(20, rows_shown)} returned).")
    lines.append("   Top examples (by severity_score):")
    for r in top_examples:
        lines.append(
            f"    - {r['cmdb_id']} | {r['kpi_name']} — anomaly_count={int(r['anomaly_count'])}, "
            f"earliest={r['earliest_anomaly_ts']}, window_min={r['window_min']}, window_max={r['window_max']}, "
            f"p95={r['p95']}, p05={r['p05']}, severity_score={r['severity_score']}"
        )
    lines.append("")
    lines.append("   Observations:")
    lines.append("    - Several DB KPIs show extremely large high-side severity (window_max >> p95), e.g.:")
    if rows_shown > 0:
        top3 = per_group.head(3).to_dict('records')
        for r in top3:
            lines.append(f"      • {r['cmdb_id']} | {r['kpi_name']} severity={r['severity_score']}")
    lines.append("    - Many of these groups have anomaly_count = 30 (continuous anomalies across the window) and earliest timestamps at the window start.")
    lines.append("")
    lines.append("2) Per-component totals (sum of anomaly_count for selected components):")
    lines.append(f"   - Components returned: {comp_rows}")
    lines.append("   - Top components by total_anomalies:")
    # show top 10 components or all if fewer
    top_components = per_component.head(10).to_dict('records')
    for c in top_components:
        lines.append(f"    - {c['cmdb_id']}: total_anomalies = {int(c['total_anomalies'])}")
    lines.append("")
    lines.append("Interpretation & recommended next steps:")
    lines.append(" - The highest-severity metric groups are primarily DB-related KPIs (examples above). These indicate large magnitude breaches in the incident window and should be prioritized.")
    lines.append(" - Many groups show continuous 30-minute anomaly runs; correlate these DB KPIs with application traces and logs to find root cause.")
    lines.append(" - Also validate metric instrumentation for any outlier numeric scales (some severity values are extremely large), to ensure thresholds and units are consistent.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id               kpi_name  anomaly_count      earliest_anomaly_ts  window_min  window_max          p95       p05  severity_score
283  db_007        SctRead_Per_Sec             30  2020-05-27 13:00:00 CST       0.000       1.140      0.00000     0.000    1.140000e+12
460  db_011            TPS_Per_Sec             30  2020-05-27 13:00:00 CST       0.000       0.200      0.00000     0.000    2.000000e+11
465  db_011            User_Commit             30  2020-05-27 13:00:00 CST       0.000       0.180      0.00000     0.000    1.800000e+11
550  db_013            User_Commit             30  2020-05-27 13:00:00 CST       0.000       0.180      0.00000     0.000    1.800000e+11
43   db_002                   AIOS             30  2020-05-27 13:00:00 CST       0.000       0.010      0.00000     0.000    1.000000e+10
255  db_007                   AIOS             30  2020-05-27 13:00:00 CST       0.000       0.010      0.00000     0.000    1.000000e+10
517  db_013    DFParaWrite_Per_Sec              5  2020-05-27 13:01:00 CST       0.000      59.950      0.06100     0.000    9.817869e+02
408  db_010           Redo_Per_Sec              4  2020-05-27 13:01:00 CST     166.500   33330.100    429.61500   172.870    7.658132e+01
536  db_013           Redo_Per_Sec              6  2020-05-27 13:01:00 CST      56.600   28712.000    396.05000    57.745    7.149590e+01
66   db_002  Physical_Read_Per_Sec             29  2020-05-27 13:00:00 CST       0.000      16.600      0.30500     0.000    5.342623e+01
451  db_011           Redo_Per_Sec              5  2020-05-27 13:00:00 CST     400.400   27528.500    664.41500   406.930    4.043269e+01
12   db_001     Logic_Read_Per_Sec              9  2020-05-27 13:00:00 CST       6.400  696461.500  28361.05000     6.600    2.355697e+01
214  db_006                   AIOS             30  2020-05-27 13:00:00 CST       0.000       0.010      0.00050     0.000    1.900000e+01
55   db_002     Logic_Read_Per_Sec              3  2020-05-27 13:00:00 CST     488.500  694794.400  76834.91500   488.355    8.042691e+00
183  db_005     Logic_Read_Per_Sec              3  2020-05-27 13:00:00 CST    2506.900   41831.400   4959.14000  2523.360    7.435213e+00
422  db_010            User_Commit             29  2020-05-27 13:00:00 CST       0.000       0.380      0.05000     0.000    6.600000e+00
436  db_011         LFSync_Per_Sec              5  2020-05-27 13:04:00 CST       0.400       3.780      0.65000     0.400    4.815385e+00
77   db_002            Session_pct             13  2020-05-27 13:00:00 CST       0.018       0.435      0.10945     0.018    2.974418e+00
50   db_002                 DbTime              4  2020-05-27 13:00:00 CST       4.700     120.900     30.62000     4.795    2.948400e+00
73   db_002            Sess_Active             13  2020-05-27 13:00:00 CST       0.000       1.200      0.31000     0.000    2.870968e+00,        cmdb_id  total_anomalies
5       db_006              678
4       db_005              675
10      db_011              651
9       db_010              612
8       db_009              595
7       db_008              574
0       db_001              558
12      db_013              555
1       db_002              555
3       db_004              531
6       db_007              497
11      db_012              494
2       db_003              445
13  docker_003              247)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
# Ensure timezone-aware timestamp column exists
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    metric_df['ts'] = pd.to_datetime(metric_df['ts']).dt.tz_convert(tz)

# 1) Compute global thresholds (P95, P05) per (cmdb_id, kpi_name) for db_* components using full series
db_mask_full = metric_df['cmdb_id'].astype(str).str.startswith('db_')
db_full = metric_df[db_mask_full].copy()

if db_full.empty:
    # No db_* series present
    db_group_anom_top50 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts',
        'window_min','window_max','p95','p05','severity_score'
    ])
    db_component_summary = pd.DataFrame(columns=['cmdb_id','total_anomalies'])
else:
    db_thresholds = (
        db_full.groupby(['cmdb_id','kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
        .reset_index()
    )

    # 2) Filter rows to incident window (inclusive start, exclusive end) in UTC+8
    start_ts = tz.localize(pd.to_datetime("2020-05-27 05:00:00"))
    end_ts = tz.localize(pd.to_datetime("2020-05-27 05:30:00"))
    db_window = db_full[(db_full['ts'] >= start_ts) & (db_full['ts'] < end_ts)].copy()

    if db_window.empty:
        # No rows in window => zero anomalies
        db_group_anom_top50 = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts',
            'window_min','window_max','p95','p05','severity_score'
        ])
        # per-component summary: list db components with zero anomalies
        unique_dbs = sorted(db_full['cmdb_id'].unique())
        db_component_summary = pd.DataFrame({'cmdb_id': unique_dbs, 'total_anomalies': 0})
    else:
        # Join thresholds into window rows (thresholds computed on full series as required)
        db_window = db_window.merge(db_thresholds, on=['cmdb_id','kpi_name'], how='left')

        # 2) Flag anomalies where value >= p95 or value <= p05
        db_window['is_anomaly'] = (db_window['value'] >= db_window['p95']) | (db_window['value'] <= db_window['p05'])

        # 3) Aggregate anomalies by (cmdb_id, kpi_name)
        # anomaly_count and earliest anomaly timestamp (from anomaly rows)
        anom_agg = (
            db_window[db_window['is_anomaly']]
            .groupby(['cmdb_id','kpi_name'])
            .agg(
                anomaly_count=('is_anomaly','sum'),
                earliest_anomaly_ts=('ts','min')
            )
            .reset_index()
        )

        if anom_agg.empty:
            # No anomalies found
            db_group_anom_top50 = pd.DataFrame(columns=[
                'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts',
                'window_min','window_max','p95','p05','severity_score'
            ])
            unique_dbs = sorted(db_window['cmdb_id'].unique())
            db_component_summary = pd.DataFrame({'cmdb_id': unique_dbs, 'total_anomalies': 0})
        else:
            # window_min/window_max per group (from all window rows)
            window_stats = (
                db_window
                .groupby(['cmdb_id','kpi_name'])['value']
                .agg(window_min='min', window_max='max')
                .reset_index()
            )

            # Merge thresholds and window stats into anomaly summary
            summary = (
                anom_agg
                .merge(db_thresholds, on=['cmdb_id','kpi_name'], how='left')
                .merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
            )

            # Compute severity_score per formula
            def compute_sev(row):
                wm = row['window_max']
                wn = row['window_min']
                p95 = row['p95']
                p05 = row['p05']
                high = 0.0
                low = 0.0
                if pd.notna(wm) and pd.notna(p95) and wm >= p95:
                    high = (wm - p95) / (abs(p95) + 1e-12)
                if pd.notna(wn) and pd.notna(p05) and wn <= p05:
                    low = (p05 - wn) / (abs(p05) + 1e-12)
                return float(max(high, low))

            summary['severity_score'] = summary.apply(compute_sev, axis=1)

            # Format earliest_anomaly_ts as string in UTC+8
            summary['earliest_anomaly_ts'] = pd.to_datetime(summary['earliest_anomaly_ts']).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

            # Round numeric columns for compactness
            for c in ['window_min','window_max','p95','p05','severity_score']:
                summary[c] = summary[c].map(lambda v: round(float(v),6) if pd.notna(v) else v)

            # Filter groups that have >=1 anomaly (already true) and order by severity desc then anomaly_count desc
            db_group_anom_top50 = summary.sort_values(['severity_score','anomaly_count'], ascending=[False, False]).head(50)
            db_group_anom_top50 = db_group_anom_top50[[
                'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts',
                'window_min','window_max','p95','p05','severity_score'
            ]]

            # 4) Per-component summary: sum anomaly_count by cmdb_id
            db_component_summary = (
                db_group_anom_top50.groupby('cmdb_id', as_index=False)['anomaly_count']
                .sum()
                .rename(columns={'anomaly_count':'total_anomalies'})
                .sort_values('total_anomalies', ascending=False)
            )

# Final compact outputs
db_group_anom_top50, db_component_summary

```
Out[7]:
```
```python
# Compose a concise plain-English summary based on the computed variables:
# db_group_anom_top50, db_component_summary (from the previous computation in the kernel).

try:
    db_groups = db_group_anom_top50
    db_comps = db_component_summary
except NameError:
    summary = "Required results (db_group_anom_top50 or db_component_summary) are not available in the kernel."
else:
    # Basic counts
    num_groups = int(db_groups.shape[0])
    num_components = int(db_comps.shape[0])
    total_anom_all = int(db_comps['total_anomalies'].sum()) if num_components > 0 else 0

    # Top 5 groups by severity (already ordered in db_group_anom_top50)
    top5 = db_groups.head(5)[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','window_min','window_max','p95','p05','severity_score']].to_dict('records')

    # Top components by total anomalies (show top 10)
    top_components = db_comps.head(10).to_dict('records')

    # Helper for compact numeric formatting
    def fmt(x):
        try:
            if pd.isna(x):
                return "NA"
        except:
            pass
        try:
            x = float(x)
            # show large severity in scientific if needed
            if abs(x) >= 1e6 or (x != 0 and abs(x) < 1e-3):
                return f"{x:.6g}"
            else:
                return f"{x:.6f}"
        except:
            return str(x)

    lines = []
    lines.append("DB metrics anomalies — incident window: 2020-05-27 05:00–05:30 (UTC+8)")
    lines.append("")
    lines.append(f"Summary:")
    lines.append(f"- Groups with >=1 anomaly returned: {num_groups} (limited to top 50 by severity).")
    lines.append(f"- DB components with anomalies: {num_components}.")
    lines.append(f"- Total anomaly points across these DB components (sum): {total_anom_all}.")
    lines.append("")
    lines.append("Top groups by severity (up to 5 shown):")
    for r in top5:
        lines.append(
            f" - {r['cmdb_id']} | {r['kpi_name']} — anomalies={int(r['anomaly_count'])}, "
            f"earliest={r['earliest_anomaly_ts']}, window_min={fmt(r['window_min'])}, window_max={fmt(r['window_max'])}, "
            f"p95={fmt(r['p95'])}, p05={fmt(r['p05'])}, severity={fmt(r['severity_score'])}"
        )
    lines.append("")
    lines.append("Observations and interpretation:")
    lines.append("- Many top groups show continuous anomalies across the full 30-minute window (anomaly_count often = 30, earliest at window start).")
    lines.append("- Several very large severity scores occur where the global P95 equals 0 (or near-zero); our severity formula divides by p95 (plus tiny epsilon), so window_max > 0 against p95==0 yields extremely large severity values. These indicate magnitude breaches but also suggest verifying thresholds/units because P95==0 can amplify severity numerically.")
    lines.append("- Examples of such cases include SctRead_Per_Sec (db_007), TPS_Per_Sec and User_Commit (db_011, db_013), and AIOS in some DBs.")
    lines.append("")
    lines.append("Per-component totals (top components by total_anomalies):")
    for c in top_components:
        lines.append(f" - {c['cmdb_id']}: total_anomalies = {int(c['total_anomalies'])}")
    lines.append("")
    lines.append("Recommended next steps:")
    lines.append("- Prioritize investigation of DB groups with large severity and continuous anomalies (examples listed above).")
    lines.append("- Check whether P95/P05 thresholds for those KPIs are meaningful (P95==0 cases should be inspected for metric sparsity or unit mismatch).")
    lines.append("- Correlate these DB KPI anomalies with application traces/logs and resource metrics to locate root causes (e.g., IO, locking, spikes in logical reads or redo activity).")
    lines.append("- If many groups are constant-equal-to-threshold (leading to many anomalies with low effective breach), verify collection/aggregation frequency and instrumentation.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id             kpi_name  anomaly_count      earliest_anomaly_ts  window_min  window_max   p95   p05  severity_score
283  db_007      SctRead_Per_Sec             30  2020-05-27 05:00:00 CST         0.0        1.14   0.0   0.0    1.140000e+12
460  db_011          TPS_Per_Sec             30  2020-05-27 05:00:00 CST         0.0        0.20   0.0   0.0    2.000000e+11
465  db_011          User_Commit             30  2020-05-27 05:00:00 CST         0.0        0.18   0.0   0.0    1.800000e+11
550  db_013          User_Commit             30  2020-05-27 05:00:00 CST         0.0        0.18   0.0   0.0    1.800000e+11
43   db_002                 AIOS             30  2020-05-27 05:00:00 CST         0.0        0.01   0.0   0.0    1.000000e+10
..      ...                  ...            ...                      ...         ...         ...   ...   ...             ...
184  db_005        Login_Per_Sec             30  2020-05-27 05:00:00 CST         0.1        0.30   0.2   0.1    5.000000e-01
242  db_006          Sess_Active             30  2020-05-27 05:00:00 CST         0.1        0.30   0.2   0.1    5.000000e-01
395  db_010        Login_Per_Sec             30  2020-05-27 05:00:00 CST         0.1        0.30   0.2   0.1    5.000000e-01
540  db_013          Sess_Active             30  2020-05-27 05:00:00 CST         0.2        0.30   0.2   0.2    5.000000e-01
551  db_013  tnsping_result_time             30  2020-05-27 05:00:00 CST        10.0       30.00  20.0  10.0    5.000000e-01

[50 rows x 9 columns],    cmdb_id  total_anomalies
2   db_005              104
10  db_013              101
1   db_002               97
3   db_006               90
8   db_011               79
7   db_010               78
4   db_007               73
0   db_001               68
9   db_012               63
5   db_008                7
6   db_009                2)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing metric_df / trace_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    metric_df['ts'] = pd.to_datetime(metric_df['ts']).dt.tz_convert(tz)

try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    trace_df['ts'] = pd.to_datetime(trace_df['ts']).dt.tz_convert(tz)

# Incident window (UTC+8)
start_ts = tz.localize(pd.to_datetime("2020-05-27 05:00:00"))
end_ts = tz.localize(pd.to_datetime("2020-05-27 05:30:00"))

# --- PART 1: Trace earliest breach for docker_003 / trace.self.jdbc.duration_p95 (use global thresholds) ---
# Ensure trace thresholds (global) exist or compute
try:
    trace_thresholds
except NameError:
    trace_thresholds = (
        trace_df.groupby(['cmdb_id','trace_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
        .reset_index()
    )

# Get the group's global thresholds
group_mask = (trace_thresholds['cmdb_id'] == 'docker_003') & (trace_thresholds['trace_name'] == 'trace.self.jdbc.duration_p95')
if group_mask.any():
    tr_p95 = float(trace_thresholds.loc[group_mask, 'p95'].iloc[0])
    tr_p05 = float(trace_thresholds.loc[group_mask, 'p05'].iloc[0])
    # Filter trace rows to group and window
    tr_window = trace_df[
        (trace_df['cmdb_id'] == 'docker_003') &
        (trace_df['trace_name'] == 'trace.self.jdbc.duration_p95') &
        (trace_df['ts'] >= start_ts) & (trace_df['ts'] < end_ts)
    ].copy()
    # Strict breach: value > p95
    tr_breach = tr_window[tr_window['value'] > tr_p95].copy()
    if not tr_breach.empty:
        # Earliest breach row
        tr_breach_earliest = tr_breach.sort_values('timestamp').iloc[0]
        trace_earliest = pd.DataFrame([{
            'timestamp': int(tr_breach_earliest['timestamp']),
            'value': float(tr_breach_earliest['value']),
            'p95': tr_p95,
            'p05': tr_p05
        }])
    else:
        trace_earliest = None
else:
    trace_earliest = None

# --- PART 2: Metrics earliest breach across all db_* groups ---
# Compute global thresholds for db_* groups if not present
db_mask_full = metric_df['cmdb_id'].astype(str).str.startswith('db_')
db_full = metric_df[db_mask_full].copy()

if db_full.empty:
    metric_earliest = None
    num_groups_severity_gt0 = 0
    total_db_anomaly_points = 0
else:
    try:
        db_thresholds
    except NameError:
        db_thresholds = (
            db_full.groupby(['cmdb_id','kpi_name'])['value']
            .agg(p95=lambda x: x.quantile(0.95), p05=lambda x: x.quantile(0.05))
            .reset_index()
        )

    # Filter db rows to incident window
    db_window = db_full[(db_full['ts'] >= start_ts) & (db_full['ts'] < end_ts)].copy()
    if db_window.empty:
        metric_earliest = None
        num_groups_severity_gt0 = 0
        total_db_anomaly_points = 0
    else:
        # Join thresholds (computed on full series) to window rows
        db_window = db_window.merge(db_thresholds, on=['cmdb_id','kpi_name'], how='left')

        # For counting anomalies (consistent with prior work), use >= p95 or <= p05
        db_window['is_anomaly_count'] = (db_window['value'] >= db_window['p95']) | (db_window['value'] <= db_window['p05'])
        total_db_anomaly_points = int(db_window['is_anomaly_count'].sum())

        # For earliest breach across all groups use strict > p95 or < p05 as requested
        db_window['is_breach_strict'] = (db_window['value'] > db_window['p95']) | (db_window['value'] < db_window['p05'])
        breaches = db_window[db_window['is_breach_strict']].copy()

        if breaches.empty:
            metric_earliest = None
        else:
            # earliest breach by timestamp (integer seconds)
            breaches_sorted = breaches.sort_values('timestamp')
            b = breaches_sorted.iloc[0]
            metric_earliest = pd.DataFrame([{
                'timestamp': int(b['timestamp']),
                'cmdb_id': b['cmdb_id'],
                'kpi_name': b['kpi_name'],
                'value': float(b['value']),
                'p95': float(b['p95']),
                'p05': float(b['p05'])
            }])

        # Compute per-(cmdb_id,kpi_name) window stats to calculate severity_score
        window_stats = (
            db_window.groupby(['cmdb_id','kpi_name'])['value']
            .agg(window_min='min', window_max='max')
            .reset_index()
        )
        groups = window_stats.merge(db_thresholds, on=['cmdb_id','kpi_name'], how='left')

        def calc_sev(row):
            wm = row['window_max']
            wn = row['window_min']
            p95 = row['p95']
            p05 = row['p05']
            high = 0.0
            low = 0.0
            if pd.notna(wm) and pd.notna(p95) and wm > p95:
                high = (wm - p95) / (abs(p95) + 1e-12)
            if pd.notna(wn) and pd.notna(p05) and wn < p05:
                low = (p05 - wn) / (abs(p05) + 1e-12)
            return max(high, low)

        groups['severity_score'] = groups.apply(calc_sev, axis=1)
        # Count distinct groups with severity_score > 0
        num_groups_severity_gt0 = int((groups['severity_score'] > 0).sum())

# Final compact outputs: trace_earliest (one-row df or None), metric_earliest (one-row df or None),
# num_groups_severity_gt0, total_db_anomaly_points
trace_earliest, metric_earliest, num_groups_severity_gt0, total_db_anomaly_points

```
Out[8]:
```
```python
# Compose plain-English summary using existing kernel results: trace_earliest, metric_earliest,
# num_groups_severity_gt0, total_db_anomaly_points

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Use variables from kernel if present
try:
    trace_earliest
except NameError:
    trace_earliest = None

try:
    metric_earliest
except NameError:
    metric_earliest = None

try:
    num_groups_severity_gt0
except NameError:
    num_groups_severity_gt0 = None

try:
    total_db_anomaly_points
except NameError:
    total_db_anomaly_points = None

lines = []
lines.append("Earliest breach findings within incident window (2020-05-27 05:00:00 to 05:30:00 UTC+8):")
lines.append("")

# Part 1: trace
if trace_earliest is None:
    lines.append("1) Trace (docker_003 | trace.self.jdbc.duration_p95): No breach found (value > global P95) in the window.")
else:
    # trace_earliest is a one-row DataFrame
    r = trace_earliest.iloc[0]
    ts_int = int(r['timestamp'])
    ts_human = pd.to_datetime(ts_int, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    lines.append("1) Trace (docker_003 | trace.self.jdbc.duration_p95):")
    lines.append(f"   - Earliest strict breach (value > global P95): timestamp = {ts_int}  (UTC+8: {ts_human})")
    lines.append(f"   - value = {float(r['value'])}, global P95 = {float(r['p95'])}, global P05 = {float(r['p05'])}")

lines.append("")

# Part 2: metrics (db_*)
if metric_earliest is None:
    lines.append("2) Metrics (all db_*): No strict breach (value > P95 or value < P05) found in the window.")
else:
    m = metric_earliest.iloc[0]
    mts_int = int(m['timestamp'])
    mts_human = pd.to_datetime(mts_int, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    lines.append("2) Metrics (db_*): earliest strict breach across all db_* groups:")
    lines.append(f"   - timestamp = {mts_int}  (UTC+8: {mts_human})")
    lines.append(f"   - cmdb_id = {m['cmdb_id']}, kpi_name = {m['kpi_name']}")
    lines.append(f"   - value = {float(m['value'])}, global P95 = {float(m['p95'])}, global P05 = {float(m['p05'])}")

lines.append("")

# Counts
lines.append("Additional counts (db_* within window):")
if num_groups_severity_gt0 is None:
    lines.append(" - distinct (cmdb_id,kpi_name) groups with severity_score > 0: N/A")
else:
    lines.append(f" - distinct (cmdb_id,kpi_name) groups with severity_score > 0: {int(num_groups_severity_gt0)}")
if total_db_anomaly_points is None:
    lines.append(" - total DB anomaly points in window: N/A")
else:
    lines.append(f" - total DB anomaly points in window: {int(total_db_anomaly_points)}")

lines.append("")
lines.append("Notes:")
lines.append("- Global thresholds (P95/P05) used were computed from each group's full available series (not recomputed on the window).")
lines.append("- For the trace check we used strict '>' against P95; for the metric earliest breach we used strict '>' or '<' against P95/P05 respectively.")
lines.append("- Timestamps shown as integer seconds and as UTC+8 human-readable times for convenience.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp  value    p95    p05
0  1590527820  0.012  0.004  0.003,     timestamp cmdb_id             kpi_name  value     p95  p05
0  1590526800  db_001  DFParaWrite_Per_Sec   2.74  2.4915  0.0, 251, 7420)```
```